## Setup do Ollama

In [17]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [18]:
import threading, subprocess

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()

In [19]:
!ollama pull qwen2.5-coder:1.5b

In [20]:
!pip install -q dspy-ai

## Configuração do DSPy

In [21]:
import dspy
import requests
import sqlite3

def setup_ollama():
    """Configure DSPy to use Ollama."""
    try:
        requests.get("http://localhost:11434/api/tags")
        print("Ollama is running")
    except Exception:
        print("Ollama not running. Start with: ollama serve")
        return False

In [22]:
dspy.settings.configure(
    lm=dspy.LM(
        model="ollama/qwen2.5-coder:1.5b",
        api_base="http://localhost:11434",
        max_tokens=200,
        temperature=0.2,
    )
)

## Banco de dados de exemplo

In [23]:
def create_sample_database():
    conn = sqlite3.connect("faculdade.db")
    c = conn.cursor()

    # zera tudo se já existir (mais fácil de iterar)
    for t in ["matriculas", "turmas", "disciplinas",
              "salas", "alunos", "professores"]:
        c.execute(f"DROP TABLE IF EXISTS {t}")

    c.executescript("""
        CREATE TABLE alunos (
            id INTEGER PRIMARY KEY,
            nome TEXT NOT NULL,
            ra TEXT UNIQUE,
            curso TEXT,
            ano_ingresso INTEGER
        );

        CREATE TABLE professores (
            id INTEGER PRIMARY KEY,
            nome TEXT NOT NULL,
            departamento TEXT,
            email TEXT,
            titulacao TEXT          -- 'Mestre', 'Doutor', 'Especialista'
        );

        CREATE TABLE disciplinas (
            id INTEGER PRIMARY KEY,
            codigo TEXT UNIQUE,
            nome TEXT NOT NULL,
            creditos INTEGER,
            departamento TEXT
        );

        CREATE TABLE salas (
            id INTEGER PRIMARY KEY,
            nome TEXT,
            predio TEXT,
            capacidade INTEGER
        );

        CREATE TABLE turmas (
            id INTEGER PRIMARY KEY,
            disciplina_id INTEGER REFERENCES disciplinas(id),
            professor_id  INTEGER REFERENCES professores(id),
            sala_id       INTEGER REFERENCES salas(id),
            semestre TEXT,          -- ex.: '2026.1'
            horario  TEXT
        );

        CREATE TABLE matriculas (
            aluno_id INTEGER REFERENCES alunos(id),
            turma_id INTEGER REFERENCES turmas(id),
            nota REAL,
            frequencia INTEGER,     -- em %
            PRIMARY KEY (aluno_id, turma_id)
        );
    """)

    c.executemany("INSERT INTO alunos VALUES (?,?,?,?,?)", [
        (1, "Ana Carolina Silva",  "20231001", "Ciência da Computação",   2023),
        (2, "Bruno Martins",       "20221045", "Ciência da Computação",   2022),
        (3, "Carla Souza",         "20231078", "Engenharia de Software",  2023),
        (4, "Diego Oliveira",      "20211015", "Ciência da Computação",   2021),
        (5, "Eduardo Lima",        "20231189", "Engenharia de Software",  2023),
        (6, "Fernanda Costa",      "20221234", "Sistemas de Informação",  2022),
        (7, "Gabriel Pereira",     "20231290", "Sistemas de Informação",  2023),
        (8, "Helena Rodrigues",    "20211067", "Ciência da Computação",   2021),
    ])

    c.executemany("INSERT INTO professores VALUES (?,?,?,?,?)", [
        (1, "Roberto Alves",     "Computação",  "roberto.alves@uni.br", "Doutor"),
        (2, "Mariana Ferreira",  "Computação",  "mariana.f@uni.br",     "Doutor"),
        (3, "Paulo Henrique",    "Matemática",  "paulo.h@uni.br",       "Mestre"),
        (4, "Sandra Nogueira",   "Computação",  "sandra.n@uni.br",      "Doutor"),
    ])

    c.executemany("INSERT INTO disciplinas VALUES (?,?,?,?,?)", [
        (1, "COMP101", "Algoritmos e Estruturas de Dados", 4, "Computação"),
        (2, "COMP201", "Banco de Dados",                   4, "Computação"),
        (3, "MAT101",  "Cálculo I",                        6, "Matemática"),
        (4, "COMP301", "Inteligência Artificial",          4, "Computação"),
        (5, "COMP202", "Engenharia de Software",           4, "Computação"),
    ])

    c.executemany("INSERT INTO salas VALUES (?,?,?,?)", [
        (1, "A101", "Bloco A", 40),
        (2, "A102", "Bloco A", 40),
        (3, "B201", "Bloco B", 60),
        (4, "Lab1", "Bloco C", 30),
        (5, "Lab2", "Bloco C", 30),
    ])

    c.executemany("INSERT INTO turmas VALUES (?,?,?,?,?,?)", [
        (1, 1, 1, 4, "2026.1", "Seg/Qua 08:00-10:00"),
        (2, 2, 2, 1, "2026.1", "Ter/Qui 10:00-12:00"),
        (3, 3, 3, 3, "2026.1", "Seg/Qua 14:00-16:00"),
        (4, 4, 4, 5, "2026.1", "Ter/Qui 14:00-16:00"),
        (5, 5, 1, 2, "2026.1", "Sex 08:00-12:00"),
    ])

    c.executemany("INSERT INTO matriculas VALUES (?,?,?,?)", [
        (1, 1, 8.5, 95),
        (1, 3, 7.0, 88),
        (2, 2, 9.0, 100),
        (2, 4, 6.5, 80),
        (3, 1, 7.5, 92),
        (4, 4, 8.0, 90),
        (4, 2, 9.5, 98),
        (5, 5, 8.0, 85),
        (6, 2, 6.0, 75),
        (7, 3, 7.0, 80),
        (8, 4, 9.0, 95),
    ])

    conn.commit()
    conn.close()
    print("Banco da faculdade criado")


In [24]:
conn = sqlite3.connect("faculdade.db")

In [25]:
SCHEMA = """
- alunos(id, nome, ra, curso, ano_ingresso)
- professores(id, nome, departamento, email, titulacao)
- disciplinas(id, codigo, nome, creditos, departamento)
- salas(id, nome, predio, capacidade)
- turmas(id, disciplina_id, professor_id, sala_id, semestre, horario)
- matriculas(aluno_id, turma_id, nota, frequencia)

Relações:
- turmas.disciplina_id  -> disciplinas.id
- turmas.professor_id   -> professores.id
- turmas.sala_id        -> salas.id
- matriculas.aluno_id   -> alunos.id
- matriculas.turma_id   -> turmas.id
"""

## Module — gerar → refinar (se falhar) → interpretar

In [26]:
class GenerateSQL(dspy.Signature):
    """Generate a SQL query to answer the question using the given schema."""
    schema: str    = dspy.InputField(desc="Database schema: tables and columns")
    question: str  = dspy.InputField(desc="Natural language question")
    sql_query: str = dspy.OutputField(desc="Valid SQL SELECT query")


class RefineSQL(dspy.Signature):
    """Fix a SQL query that produced an error."""
    schema: str      = dspy.InputField(desc="Database schema: tables and columns")
    question: str    = dspy.InputField(desc="Natural language question")
    sql_query: str   = dspy.InputField(desc="SQL query that failed")
    error: str       = dspy.InputField(desc="Error message from the failed query")
    refined_sql: str = dspy.OutputField(desc="Corrected SQL query")


class InterpretResults(dspy.Signature):
    """Answer the question in natural language based on SQL results."""
    question: str  = dspy.InputField(desc="Original question")
    sql_query: str = dspy.InputField(desc="SQL query that was executed")
    results: str   = dspy.InputField(desc="Rows returned by the query")
    answer: str    = dspy.OutputField(desc="Human-readable answer")

/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "GenerateSQL" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)
/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "RefineSQL" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)


In [27]:
def _clean(sql: str) -> str:
    """Remove cercas markdown e ; finais que o LLM às vezes inclui."""
    sql = sql.strip()
    if sql.startswith("```"):
        sql = sql.strip("`")
        if sql.lower().startswith("sql"):
            sql = sql[3:]
    return sql.strip().rstrip(";")


class SQLGenerator(dspy.Module):
    """Gera, executa, refina em caso de erro e interpreta o resultado."""

    def __init__(self, schema=SCHEMA):
        super().__init__()
        self.schema      = schema
        self.generator   = dspy.ChainOfThought(GenerateSQL)
        self.refiner     = dspy.ChainOfThought(RefineSQL)
        self.interpreter = dspy.Predict(InterpretResults)

    def forward(self, question, conn):
        sql = _clean(self.generator(schema=self.schema, question=question).sql_query)

        try:
            results = conn.execute(sql).fetchall()
        except Exception as e:
            ref = self.refiner(schema=self.schema, question=question,
                               sql_query=sql, error=str(e))
            sql = _clean(ref.refined_sql)
            try:
                results = conn.execute(sql).fetchall()
            except Exception as e2:
                return dspy.Prediction(sql_query=sql, results=None,
                                       answer=None, error=str(e2))

        answer = self.interpreter(question=question, sql_query=sql,
                                  results=str(results)).answer
        return dspy.Prediction(sql_query=sql, results=results,
                               answer=answer, error=None)

## Exemplos de treino

In [28]:
def create_examples():
    return [
        dspy.Example(
            question="Quantos alunos estão matriculados em Ciência da Computação?",
            sql_query="SELECT COUNT(*) FROM alunos WHERE curso = 'Ciência da Computação'"
        ).with_inputs("question"),
        dspy.Example(
            question="Qual a média das notas da disciplina de Banco de Dados?",
            sql_query="""SELECT AVG(m.nota) FROM matriculas m
                         JOIN turmas t ON t.id = m.turma_id
                         JOIN disciplinas d ON d.id = t.disciplina_id
                         WHERE d.nome = 'Banco de Dados'"""
        ).with_inputs("question"),
        dspy.Example(
            question="Liste os professores doutores",
            sql_query="SELECT nome, departamento FROM professores WHERE titulacao = 'Doutor'"
        ).with_inputs("question"),
        dspy.Example(
            question="Quais salas têm capacidade maior que 40?",
            sql_query="SELECT nome, predio, capacidade FROM salas WHERE capacidade > 40"
        ).with_inputs("question"),
        dspy.Example(
            question="Quais alunos tiraram nota acima de 8?",
            sql_query="""SELECT a.nome, m.nota FROM matriculas m
                         JOIN alunos a ON a.id = m.aluno_id
                         WHERE m.nota > 8"""
        ).with_inputs("question"),
        dspy.Example(
            question="Em qual sala é a aula de Inteligência Artificial?",
            sql_query="""SELECT s.nome, s.predio, t.horario FROM turmas t
                         JOIN disciplinas d ON d.id = t.disciplina_id
                         JOIN salas s ON s.id = t.sala_id
                         WHERE d.nome = 'Inteligência Artificial'"""
        ).with_inputs("question"),
    ]

In [29]:
setup_ollama()
create_sample_database()
conn = sqlite3.connect("faculdade.db")

Ollama is running
Banco da faculdade criado


## Otimizador — BootstrapFewShot

In [30]:
def sql_match_metric(example, pred, trace=None):
    """Linhas previstas == linhas do SQL gabarito do exemplo."""
    if pred is None or pred.results is None:
        return False
    try:
        gold = conn.execute(example.sql_query).fetchall()
    except Exception:
        return False
    return sorted(map(str, pred.results)) == sorted(map(str, gold))


class SQLGeneratorForOpt(dspy.Module):
    """Wrapper que fixa `conn` para que o módulo aceite só `question` no forward."""
    def __init__(self, conn, schema=SCHEMA):
        super().__init__()
        self.conn  = conn
        self.inner = SQLGenerator(schema)

    def forward(self, question):
        return self.inner(question=question, conn=self.conn)

In [31]:
from dspy.teleprompt import BootstrapFewShot

trainset = create_examples()
print("Compilando programa com BootstrapFewShot...")
optimizer = BootstrapFewShot(
    metric=sql_match_metric,
    max_bootstrapped_demos=3,
    max_labeled_demos=4,
)
compiled = optimizer.compile(SQLGeneratorForOpt(conn), trainset=trainset)
print("\u2713 Programa otimizado")

/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "StringSignature" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)


Compilando programa com BootstrapFewShot...


100%|██████████| 6/6 [00:01<00:00,  4.15it/s]

Bootstrapped 1 full traces after 5 examples for up to 1 rounds, amounting to 6 attempts.
✓ Programa otimizado


## Avaliação

In [ ]:
test_questions = [
    "Quantos alunos estão matriculados em Ciência da Computação?",
    "Qual a média das notas da disciplina de Banco de Dados?",
    "Quais alunos tiraram nota acima de 8?",
    "Em qual sala é a aula de Inteligência Artificial?",
    "Quem dá aula de Cálculo I?",
    "Quais disciplinas têm 6 créditos?",
    "Qual aluno tem a maior frequência?",
]

for i, question in enumerate(test_questions, 1):
    print(f"\nQuery {i}: {question}")
    result = compiled(question=question)
    print(f"SQL: {result.sql_query}")
    if result.error:
        print(f"Erro: {result.error}")
    else:
        print(f"Linhas: {result.results}")
        print(f"Resposta: {result.answer}")


Query 1: Quantos alunos estão matriculados em Ciência da Computação?
SQL: SELECT COUNT(DISTINCT a.id) AS numero_alunos FROM disciplinas d
JOIN turmas t ON d.id = t.disciplina_id
JOIN matriculas m ON t.id = m.turma_id
JOIN alunos a ON m.aluno_id = a.id
WHERE d.nome = 'Ciência da Computação'
Linhas: [(0,)]
Resposta: Não há alunos matriculados em Ciência da Computação.

Query 2: Qual a média das notas da disciplina de Banco de Dados?
SQL: SELECT AVG(m.nota) AS media FROM disciplinas d
JOIN turmas t ON d.id = t.disciplina_id
JOIN matriculas m ON t.id = m.turma_id
JOIN alunos a ON m.aluno_id = a.id
WHERE d.nome = 'Banco de Dados'
Linhas: [(8.166666666666666,)]
Resposta: A média das notas da disciplina de Banco de Dados é de 8.17.

Query 3: Quais alunos tiraram nota acima de 8?
SQL: SELECT a.nome FROM alunos a
JOIN matriculas m ON a.id = m.aluno_id
JOIN turmas t ON m.turma_id = t.id
JOIN disciplinas d ON t.disciplina_id = d.id
WHERE m.nota > 8
Linhas: [('Ana Carolina Silva',), ('Bruno Marti

## Bot Telegram

In [ ]:
!pip install -q python-telegram-bot nest_asyncio

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from telegram import Update
from telegram.constants import ChatAction
from telegram.ext import (Application, CommandHandler, MessageHandler,
                          filters, ContextTypes)

TELEGRAM_TOKEN = "token"

BOAS_VINDAS = (
    "Olá! Sou o bot da faculdade.\n"
    "Pergunta o que quiser sobre alunos, professores, turmas, salas, "
    "disciplinas e notas — eu traduzo pra SQL e te respondo.\n\n"
    "Comandos:\n"
    "/help — exemplos de perguntas\n"
    "/schema — mostra as tabelas do banco"
)

EXEMPLOS = (
    "*Exemplos que funcionam bem:*\n"
    "• Quantos alunos tem em Engenharia de Software?\n"
    "• Qual a média de notas em Banco de Dados?\n"
    "• Quais professores são doutores?\n"
    "• Em qual sala é a aula de IA?\n"
    "• Quais alunos tiraram nota acima de 8?\n"
    "• Quem dá aula de Cálculo I?"
)


async def start(update, ctx):
    await update.message.reply_text(BOAS_VINDAS)


async def ajuda(update, ctx):
    await update.message.reply_text(EXEMPLOS, parse_mode="Markdown")


async def mostra_schema(update, ctx):
    await update.message.reply_text(f"```\n{SCHEMA.strip()}\n```",
                                    parse_mode="Markdown")


def formata_resposta(r):
    if r.error:
        return f"❌ Não consegui responder.\n_SQL tentado:_ `{r.sql_query}`\n_Erro:_ {r.error}"
    if not r.results:
        return f"Nenhum resultado encontrado.\n_SQL:_ `{r.sql_query}`"
    return f"{r.answer}\n\n_SQL usado:_ `{r.sql_query}`"


async def responder(update, ctx):
    pergunta = update.message.text
    await ctx.bot.send_chat_action(chat_id=update.effective_chat.id,
                                   action=ChatAction.TYPING)
    try:
        r = compiled(question=pergunta)
        msg = formata_resposta(r)
    except Exception as e:
        msg = f"Erro inesperado: {e}"
    await update.message.reply_text(msg[:4000], parse_mode="Markdown")


app = Application.builder().token(TELEGRAM_TOKEN).build()
app.add_handler(CommandHandler("start", start))
app.add_handler(CommandHandler("help", ajuda))
app.add_handler(CommandHandler("schema", mostra_schema))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, responder))
print("Bot no ar, mande mensagens pelo telegram.")
app.run_polling()